In [ ]:
import kagglehub
import os
import pandas as pd



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
dataset = pd.read_csv(os.path.join(path, "Q3_data.csv"))


In [ ]:
# Task 2: Write your code here:
dataset.head()

In [ ]:
# Task 3: Write your code here:
dataset.info()

In [ ]:
# Task 4: Write your code here:
dataset.describe()

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")
check_missing_values(dataset)

In [ ]:
dataset.shape

In [ ]:
# Task 1: Write your code here:
#df_clean = dataset.dropna(subset=dataset.columns)
df_clean=dataset.copy()
for col in dataset.columns:
    df_clean[col] = dataset[col].fillna(dataset[col].mean())


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns
categorical_cols
# no col's need encoding

In [ ]:
# Task 4: Write your code here:
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df_clean[numerical_cols] = scaler.fit_transform(X)


In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  df[target_column].value_counts().plot(kind='bar')
  plt.show()
check_target_imbalance(df_clean, "Target")

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Target", axis=1).astype(float)
y = df_clean['Target'].astype(float)

In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q


In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

     # Train and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    f1 = f1_score(y_test, y_pred, zero_division=0)

In [ ]:
# Task 1: Write your code here:
feature_cols=dataset.columns.drop(['Target'])
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# then use bar to plot it
plt.barh(feature_importance['feature'], feature_importance['importance'])

In [ ]:
# Task 2: Write your code here:
feature_importance['feature'][0]

In [ ]:
# Task Bonus: Write your code here: